In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the raw dataset from the data directory
file_path = "../data/20260512_FintechTrainee_CaseStudy_Dataset_FinSightLanka_F.csv"
df = pd.read_csv(file_path)

In [3]:
# Display initial dataset shape and first few records
print("Original Data Shape (Rows, Columns):", df.shape)
display(df.head())

Original Data Shape (Rows, Columns): (508, 44)


,Customer_ID,Full_Name,NIC_Number,Gender,Date_of_Birth,Age,District,Province,Urban_Rural,Customer_Segment,...,PEP_Flag,Has_Fixed_Deposit,FD_Amount,Has_Insurance,Has_Mobile_Wallet,Number_of_Products,Acquisition_Channel,Referral_Source,Last_Campaign_Response,Promo_Code_Used
0,FSL-0001,Mark Johnson,808614657V,Female,1980-12-26,43,Kandy,Central,Suburban,Regular,...,No,No,NaN,No,Yes,2,Referral,Friend,Responded,Yes
1,FSL-0002,Deborah Phillips,718478527V,Female,1971-12-13,52,Badulla,Uva,Urban,Starter,...,No,No,NaN,No,Yes,2,Referral,Family,Ignored,No
2,FSL-0003,Charles Phillips,933515010V,Male,1993-12-17,30,Kandy,Central,Suburban,Regular,...,No,Yes,1166599.45,No,No,3,Agent,Colleague,Responded,No
3,FSL-0004,Rachel Hall,720422796V,Male,1972-02-11,51,Kandy,Central,Suburban,Regular,...,No,Yes,1051113.61,Yes,No,4,Referral,NaN,Ignored,Yes
4,FSL-0005,Sharon Bell,940729689V,Male,1994-03-13,29,Colombo,Western,Suburban,Regular,...,No,No,NaN,No,Yes,3,Referral,NaN,Unsubscribed,No


In [4]:
# check for the missing values
print(df.isnull().sum()) 



Customer_ID                    0
Full_Name                      8
NIC_Number                     0
Gender                         0
Date_of_Birth                  0
Age                            0
District                       5
Province                       0
Urban_Rural                    0
Customer_Segment               5
Account_Open_Date              0
Customer_Tenure_Years          0
Savings_Balance                6
Monthly_Deposit_Avg            6
Monthly_Withdrawal_Avg         0
Net_Monthly_Flow               0
Account_Status                 0
Interest_Rate_Tier             0
Has_Loan                       5
Loan_Type                    290
Loan_Amount                  291
Loan_Tenure_Months           290
Monthly_Repayment_Amount     290
Loan_Repayment_Status        288
Loan_Start_Date              288
Outstanding_Loan_Balance     289
Mobile_App_User                0
Last_Login_Date              169
Monthly_Transaction_Count      0
Preferred_Channel              0
Push_Notif

In [5]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 44 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Customer_ID                508 non-null    str    
 1   Full_Name                  500 non-null    str    
 2   NIC_Number                 508 non-null    str    
 3   Gender                     508 non-null    str    
 4   Date_of_Birth              508 non-null    str    
 5   Age                        508 non-null    int64  
 6   District                   503 non-null    str    
 7   Province                   508 non-null    str    
 8   Urban_Rural                508 non-null    str    
 9   Customer_Segment           503 non-null    str    
 10  Account_Open_Date          508 non-null    str    
 11  Customer_Tenure_Years      508 non-null    float64
 12  Savings_Balance            502 non-null    float64
 13  Monthly_Deposit_Avg        502 non-null    float64
 14  Month

In [6]:
# Drop rows only where critical fields (Name, Primary Financial Metrics) are missing
critical_cols = ['Full_Name', 'Savings_Balance', 'Monthly_Deposit_Avg', 'District', 'Customer_Segment']
df = df.dropna(subset=critical_cols)

In [7]:
# Fill Loan & FD missing values appropriately instead of dropping
loan_numeric_cols = ['Loan_Amount', 'Monthly_Repayment_Amount', 'Outstanding_Loan_Balance', 'Loan_Tenure_Months']
df[loan_numeric_cols] = df[loan_numeric_cols].fillna(0)
df['Loan_Type'] = df['Loan_Type'].fillna('No Loan')
df['Loan_Repayment_Status'] = df['Loan_Repayment_Status'].fillna('N/A')
df['FD_Amount'] = df['FD_Amount'].fillna(0)
df['Referral_Source'] = df['Referral_Source'].fillna('Direct/None')

In [8]:
#  Standardize Gender values (Convert 'M' -> 'Male' and 'F' -> 'Female')
if 'Gender' in df.columns:
    df['Gender'] = df['Gender'].replace({'M': 'Male', 'F': 'Female'})

#  Correct negative values in Savings Balance to absolute values
if 'Savings_Balance' in df.columns:
    df['Savings_Balance'] = df['Savings_Balance'].abs()

# Remove duplicate records based on Customer_ID
if 'Customer_ID' in df.columns:
    df = df.drop_duplicates(subset=['Customer_ID'])

#  Fix Has_Loan based on Loan_Amount (If Loan_Amount > 0 then Yes, else No)
if 'Has_Loan' in df.columns:
    df['Has_Loan'] = np.where(df['Loan_Amount'] > 0, 'Yes', 'No')

#  Fill missing KYC_Status with 'Pending'
if 'KYC_Status' in df.columns:
    df['KYC_Status'] = df['KYC_Status'].fillna('Pending')

#  Convert Loan_Start_Date and Last_Login_Date to datetime format
#  errors='coerce' will keep missing/invalid dates safely as NaT (Not a Time)
df['Loan_Start_Date'] = pd.to_datetime(df['Loan_Start_Date'], errors='coerce')
df['Last_Login_Date'] = pd.to_datetime(df['Last_Login_Date'], errors='coerce')


In [9]:
# Drop unnecessary PII (Personally Identifiable Information) columns if they exist
columns_to_drop = ['Full_Name', 'NIC_Number']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

In [10]:
# check for the missing values
print(df.info()) 

<class 'pandas.DataFrame'>
Index: 470 entries, 0 to 499
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Customer_ID                470 non-null    str           
 1   Gender                     470 non-null    str           
 2   Date_of_Birth              470 non-null    str           
 3   Age                        470 non-null    int64         
 4   District                   470 non-null    str           
 5   Province                   470 non-null    str           
 6   Urban_Rural                470 non-null    str           
 7   Customer_Segment           470 non-null    str           
 8   Account_Open_Date          470 non-null    str           
 9   Customer_Tenure_Years      470 non-null    float64       
 10  Savings_Balance            470 non-null    float64       
 11  Monthly_Deposit_Avg        470 non-null    float64       
 12  Monthly_Withdrawal_Avg  

In [11]:
# Export the cleaned dataset to a new CSV file inside the data folder
cleaned_file_path = "../data/Cleaned_FinSightLanka_Data.csv"
df.to_csv(cleaned_file_path, index=False)

print("\n **** Data Cleaning Completed successfully! ****")
print(f"Cleaned dataset saved as: '{cleaned_file_path}'")
print("Cleaned Data Shape:", df.shape)


 **** Data Cleaning Completed successfully! ****
Cleaned dataset saved as: '../data/Cleaned_FinSightLanka_Data.csv'
Cleaned Data Shape: (470, 42)
